## Define Scenarios

This notebook prepares the canonical Phase 2 handoff data for Phase 3.

The objective is to build one aligned hourly input table containing:

- actual PV power
- forecast PV power 
- load demand

This table will be used to define dispatch-visible scenarios and policy comparisons.

## Imports

In [13]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## Load the Three Phase 2 Files and Inspect the Structure

In [14]:
pv_actual = pd.read_csv("../data/raw/pv_actual_kw.csv")
pv_forecast = pd.read_csv("../data/raw/pv_forecast_kw.csv")
load = pd.read_csv("../data/raw/load_kw.csv")

print("pv_actual columns:", pv_actual.columns.tolist())
print("pv_forecast columns:", pv_forecast.columns.tolist())
print("load columns:", load.columns.tolist())

print("\npv_actual head:")
print(pv_actual.head(10))

print("\npv_forecast head:")
print(pv_forecast.head(10))

print("\nload head:")
print(load.head(10))

pv_actual columns: ['pv_actual_kw']
pv_forecast columns: ['pv_forecast_kw']
load columns: ['time', 'load_kw']

pv_actual head:
   pv_actual_kw
0      0.000000
1      0.000000
2      0.000000
3      0.000000
4      0.000000
5      0.000000
6      0.000000
7     22.492022
8     46.646185
9     66.077123

pv_forecast head:
   pv_forecast_kw
0        0.000000
1        0.000000
2        0.000000
3        0.000000
4        0.000000
5        0.000000
6        0.000000
7       29.015230
8       59.371083
9       76.839543

load head:
                        time    load_kw
0  2025-01-01 00:00:00+00:00  20.182830
1  2025-01-01 01:00:00+00:00  19.426256
2  2025-01-01 02:00:00+00:00  20.758769
3  2025-01-01 03:00:00+00:00  21.819825
4  2025-01-01 04:00:00+00:00  22.388684
5  2025-01-01 05:00:00+00:00  27.946188
6  2025-01-01 06:00:00+00:00  37.117944
7  2025-01-01 07:00:00+00:00  44.280344
8  2025-01-01 08:00:00+00:00  47.971394
9  2025-01-01 09:00:00+00:00  43.563999


## Merge into One Canonical Table

In [27]:
df = pv_actual.merge(
    pv_forecast[["pv_forecast_kw"]],
    left_index=True,
    right_index=True,
    how="inner"
)

load_data = load["load_kw"]

df = df.copy()
df['load_kw'] = load_data

df["time"] = load["time"]
df = df.set_index('time')

df = df.sort_index().reset_index()

print(df.head(10))
print(df.columns.tolist())
print("Rows:", len(df))

                        time  pv_actual_kw  pv_forecast_kw    load_kw
0  2025-01-01 00:00:00+00:00      0.000000        0.000000  20.182830
1  2025-01-01 01:00:00+00:00      0.000000        0.000000  19.426256
2  2025-01-01 02:00:00+00:00      0.000000        0.000000  20.758769
3  2025-01-01 03:00:00+00:00      0.000000        0.000000  21.819825
4  2025-01-01 04:00:00+00:00      0.000000        0.000000  22.388684
5  2025-01-01 05:00:00+00:00      0.000000        0.000000  27.946188
6  2025-01-01 06:00:00+00:00      0.000000        0.000000  37.117944
7  2025-01-01 07:00:00+00:00     22.492022       29.015230  44.280344
8  2025-01-01 08:00:00+00:00     46.646185       59.371083  47.971394
9  2025-01-01 09:00:00+00:00     66.077123       76.839543  43.563999
['time', 'pv_actual_kw', 'pv_forecast_kw', 'load_kw']
Rows: 8735


In [ ]:
fg